# Data Flywheel / Agent-in-the-Loop | Agent Infrastructure

In [1]:
# Data Flywheel: Collecting and Using Feedback
from typing import List, Dict
from dataclasses import dataclass, field

In [2]:
@dataclass
class InteractionRecord:
    query: str
    response: str
    rating: float  # 1-10
    feedback: str = ""

class DataFlywheel:
    def __init__(self):
        self.interactions: List[InteractionRecord] = []
        self.prompt_improvements: List[str] = []

    def record(self, record: InteractionRecord):
        self.interactions.append(record)

    def get_training_data(self, min_rating: float = 8.0) -> List[Dict]:
        """Extract high-quality interactions for fine-tuning."""
        return [
            {"input": r.query, "output": r.response}
            for r in self.interactions if r.rating >= min_rating
        ]

    def get_failure_patterns(self, max_rating: float = 5.0) -> List[Dict]:
        """Identify common failure patterns for prompt improvement."""
        return [
            {"query": r.query, "rating": r.rating, "feedback": r.feedback}
            for r in self.interactions if r.rating <= max_rating
        ]

    def stats(self) -> Dict:
        if not self.interactions:
            return {"count": 0}
        ratings = [r.rating for r in self.interactions]
        return {
            "count": len(self.interactions),
            "avg_rating": sum(ratings) / len(ratings),
            "high_quality_count": len(self.get_training_data()),
            "failure_count": len(self.get_failure_patterns()),
        }

In [3]:
flywheel = DataFlywheel()
flywheel.record(InteractionRecord("How to reset password?", "Go to Settings > Security...", 9.0))
flywheel.record(InteractionRecord("Complex billing issue", "I'm not sure...", 3.0, "Didn't resolve"))
flywheel.record(InteractionRecord("API documentation", "Here are the endpoints...", 8.5))

print(f"Stats: {flywheel.stats()}")
print(f"Training data: {len(flywheel.get_training_data())} examples")
print(f"Failures to analyze: {flywheel.get_failure_patterns()}")

# --- CLOSE THE LOOP: build an improved prompt from production data ---
GENERIC_PROMPT = "You are a helpful support agent. Answer the user's question."

high_quality = flywheel.get_training_data(min_rating=8.0)
few_shot_block = "\n\n".join(
    f"Example {i+1}:\n  User: {ex['input']}\n  Agent: {ex['output']}"
    for i, ex in enumerate(high_quality)
)
IMPROVED_PROMPT = (
    "You are a helpful support agent. Answer the user's question.\n\n"
    "Here are examples of highly-rated responses from production:\n"
    f"{few_shot_block}\n\n"
    "Follow the style and thoroughness shown in these examples."
)

print(f"\nBefore flywheel: generic prompt ({len(GENERIC_PROMPT)} chars, 0 examples)")
print(f"After flywheel:  prompt with {len(high_quality)} few-shot examples from production ({len(IMPROVED_PROMPT)} chars)")
print(f"Few-shot examples:\n{few_shot_block}")

Stats: {'count': 3, 'avg_rating': 6.833333333333333, 'high_quality_count': 2, 'failure_count': 1}
Training data: 2 examples
Failures to analyze: [{'query': 'Complex billing issue', 'rating': 3.0, 'feedback': "Didn't resolve"}]

Before flywheel: generic prompt (60 chars, 0 examples)
After flywheel:  prompt with 2 few-shot examples from production (335 chars)
Few-shot examples:
Example 1:
  User: How to reset password?
  Agent: Go to Settings > Security...

Example 2:
  User: API documentation
  Agent: Here are the endpoints...
